# AI-Powered Image Editing Platform: Prompt Engineering & API Handling
### Week 2 Milestone Deliverable: AI Image Editing & Version History

This notebook demonstrates:
1. **Multi-Modal Vision Analysis & Auto-Captioning** using GPT-4o / Gemini Vision.
2. **The Prompt Engineering Pipeline**: Converting casual natural language user edits into photorealistic generation specifications.
3. **Comparison**: Naive raw prompt vs. engineered prompt with subject preservation.
4. **Natural Language Semantic Search**: Executing queries like *'show me the ones with a beach'*.
5. **Version History & Lineage Tracking**: Creating, thumbnailing, and rolling back edit revisions.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from PIL import Image
from backend.storage import StorageManager
from backend.prompt_templates import (
    build_edit_synthesis_prompt,
    EDIT_PROMPT_SYNTHESIZER_SYSTEM,
    PRESET_PROMPT_ENGINEERED
)
from backend.edit_engine import synthesize_prompt_with_llm, EditEngine
from backend.search_engine import SearchEngine

print("Environment initialized successfully!")
print(f"Base Directory: {BASE_DIR}")

## 1. Inspecting Visual Context from the Catalog
The storage manager maintains rich scene metadata extracted from vision models (GPT-4o / Gemini).

In [ ]:
storage = StorageManager()
images = storage.get_all_images()
print(f"Total images in library: {len(images)}\n")

sample_img = images[0]
print(f"Title: {sample_img['title']}")
print(f"Caption: {sample_img['caption']}")
print(f"Tags: {sample_img['tags'][:6]}")
print(f"Detected Objects: {sample_img['objects']}")
print(f"Active Version: {sample_img['current_version_id']}")

## 2. Naive Prompt vs. Engineered Prompt
When a user types **'remove the background'**, sending this raw string to an image generation model fails because the model lacks scene context. Our prompt synthesizer bridges this gap.

In [ ]:
test_user_instruction = "remove the background and isolate the subject"

# Direct naive prompt
naive_prompt = test_user_instruction

# Engineered prompt combining scene context and optical guidelines
synthesis_result = synthesize_prompt_with_llm(
    user_instruction=test_user_instruction,
    image_caption=sample_img["caption"],
    objects_list=sample_img["objects"]
)

print("=== NAIVE RAW PROMPT ===")
print(naive_prompt)
print("\n=== ENGINEERED PROMPT SPECIFICATION ===")
print(f"Category: {synthesis_result.get('edit_category')}")
print(f"Summary: {synthesis_result.get('summary_of_changes')}")
print(f"Engineered Prompt: {synthesis_result.get('engineered_prompt')}")
print(f"Negative Constraints: {synthesis_result.get('negative_prompt')}")

## 3. Object Addition & Placement Prompt Synthesis
Testing user instructions like *'add a tree in the background'* or *'add a vintage camera on the table'*.

In [ ]:
addition_instruction = "add a lush green tree in the background with soft dappled sunlight"

synth_addition = synthesize_prompt_with_llm(
    user_instruction=addition_instruction,
    image_caption=sample_img["caption"],
    objects_list=sample_img["objects"]
)

print("Engineered Prompt for Object Addition:")
print(synth_addition["engineered_prompt"])

## 4. Natural Language Image Search
Evaluating queries like *'show me the ones with a beach'* or *'photos with a dog'* against our library.

In [ ]:
search_engine = SearchEngine()

test_queries = [
    "show me the ones with a beach",
    "photos with a dog in a park",
    "warm golden hour portraits",
    "cozy coffee with notebook"
]

for q in test_queries:
    print(f"\n>>> Query: '{q}'")
    results = search_engine.search(q, images, top_k=2)
    for rank, (img, score, reasons) in enumerate(results, 1):
        print(f"  Rank #{rank}: {img['title']} | Relevance Score: {score:.3f} | Matches: {reasons}")

## 5. Non-Destructive Version History & Lineage
Demonstrating how edit revisions are tracked with thumbnails, timestamps, and rollback capability.

In [ ]:
for img in images:
    print(f"Image: {img['title']} (ID: {img['id']})")
    for ver in img.get("versions", []):
        is_active = (ver["version_id"] == img["current_version_id"])
        marker = "[*] ACTIVE" if is_active else "[ ]"
        print(f"   {marker} v{ver['version_number']}: {ver['prompt']} ({ver['model_used']}) at {ver['timestamp']}")
    print("-" * 60)